# Modelagem e Avaliação de Redes Neurais

### Objetivo

- Treinar uma Rede Neural simples utilizando o MLPClassifier
- Aplicar Validação Cruzada
- Comparar o desempenho dos modelos e escolher o melhor baseado no contexto
- Exportar resultados e modelo final

### Metas

- ROC-AUC maior ou igual a 0,80
- F1-score maior ou igual a 0,60
- recall maior ou igual a 0,55

## Sumário
- [Importando Bibliotecas](#Importando-Bibliotecas)
- [Pré-processamento](#preprocessamento)
    - [Separando Features e Target](#Separando-Features-e-Target)
    - [Separando dados Númericos e Categóricos](#num-cat)
- [Construindo os Pipelines](#Construindo-os-Pipelines)
    - [Validação Cruzada](#cv)
    - [Funções de Desempenho](#desempenho)
    - [Baseline](#Baseline)
    - [Threshold ajustado](#Threshold-ajustado)
    - [Undersampling](#Undersampling)
    - [Oversampling](#Oversampling)
    - [SMOTE](#SMOTE)
- [Conclusão](#conclusao)
    - [Recomendação](#recomendacao)
    - [Exportação](#exportacao)
        - [Modelo](#Modelo)
        - [Resultados](#Resultados)


## Importando bibliotecas

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

Salvando RANDOM_STATE para permitir reprodutibilidade:

In [2]:
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if "notebooks" in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

pd.set_option("display.max_columns", None)

<h2 id="preprocessamento">Pré-processamento</h2>

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Verificamos que os dados estão com os tipos corretos. Agora, vou separar as **features** do **target**

In [7]:
balanceamento_classes = pd.DataFrame({
    "Quantidade": df["Churn"].value_counts(),
    "Percentual": df["Churn"].value_counts(normalize=True).mul(100).round(2),
})

balanceamento_classes

,Quantidade,Percentual
Churn,,
No,5174,73.46
Yes,1869,26.54


Como existe um desbalanceamento, podemos testar algumas estratégias para melhorar o desempenho do modelo levando em consideração a necessidade do negócio

### Separando Features e Target

In [8]:
X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

<h3 id="num-cat">Separando dados Númericos e Categóricos</h3>

In [9]:
cols_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]
cols_categoricas = [col for col in X.columns if col not in cols_numericas]

## Construindo os Pipelines

In [10]:
pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipeline_categorico = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessador = ColumnTransformer([
    ("num", pipeline_numerico, cols_numericas),
    ("cat", pipeline_categorico, cols_categoricas),
])

mlp = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="relu",
    solver="adam",
    max_iter=1000,
    early_stopping=True,
    random_state=RANDOM_STATE,
)

mlp_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", mlp),
])

metricas = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

<h3 id="cv">Validação Cruzada</h3>

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

<h3 id="desempenho">Funções de Desempenho</h3>

Para não repetir o mesmo código ao longo do treinamento dos modelo com técnicas de sampling, criei duas funções:
- calcular_metricas
- cv_resultados

**_calcular_metricas_** vai retornar as métricas para avaliarmos os modelos e **_cv_resultados_** vai fazer a validação cruzada considerando thresholds diversos caso seja preciso. Por padrão o limiar é 0.5 

In [12]:
def calcular_metricas(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

In [13]:
def cv_resultados(pipeline, threshold=0.5):
    metricas = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        proba = pipeline.predict_proba(X_test)[:, 1]
        preds = (proba >= threshold).astype(int)
        metricas.append(calcular_metricas(y_test, preds, proba))

    media_metricas = pd.DataFrame(metricas).mean()
    media_metricas["threshold"] = threshold
    return media_metricas

### Baseline

In [14]:
# Baseline
baseline_pipeline = Pipeline([
    ("preprocessador", preprocessador),
    ("classificador", mlp),
])

In [15]:
baseline_res = cv_resultados(baseline_pipeline)

### Threshold ajustado

In [16]:
# Threshold ajustado
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]
resultado = []

for thr in thresholds:
    resultado.append(cv_resultados(baseline_pipeline, threshold=thr))

threshold_res = resultado

### Undersampling

In [17]:
# Random Undersampling
undersample_pipe = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
    ("classificador", mlp),
])

undersample_res = cv_resultados(undersample_pipe)

### Oversampling

In [18]:
# Random Oversampling
oversample_pipe = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
    ("classificador", mlp),
])
oversample_res = cv_resultados(oversample_pipe)

### SMOTE

In [19]:
# SMOTE
smote_pipe = ImbPipeline([
    ("preprocessador", preprocessador),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("classificador", mlp),
])
smote_res = cv_resultados(smote_pipe)

<h2 id="conclusao">Conclusão</h2>

Agora, vamos comparar os resultados obtidos na tabela abaixo

In [20]:
resumo = pd.DataFrame([
    {"estrategia": "baseline", **baseline_res},
    {"estrategia": "threshold_0.30", **threshold_res[0]},
    {"estrategia": "threshold_0.35", **threshold_res[1]},
    {"estrategia": "threshold_0.40", **threshold_res[2]},
    {"estrategia": "threshold_0.45", **threshold_res[3]},
    {"estrategia": "undersampling", **undersample_res},
    {"estrategia": "oversampling", **oversample_res},
    {"estrategia": "smote", **smote_res},
])
resumo.drop(columns='threshold', inplace=True)

resumo.head(10)

,estrategia,accuracy,precision,recall,f1,roc_auc
0,baseline,0.803206,0.655575,0.543022,0.592969,0.843306
1,threshold_0.30,0.758482,0.532224,0.745809,0.620508,0.843306
2,threshold_0.35,0.777081,0.565149,0.697649,0.623464,0.843306
3,threshold_0.40,0.788155,0.592015,0.651105,0.619265,0.843306
4,threshold_0.45,0.799656,0.627600,0.602410,0.613622,0.843306
5,undersampling,0.753509,0.524748,0.775805,0.625835,0.841632
6,oversampling,0.738887,0.506155,0.768302,0.609866,0.837914
7,smote,0.745277,0.513035,0.790786,0.622159,0.842209


<h3 id="recomendacao">Recomendação</h3>

Apesar do resultado da Regressão Logística (notebook anterior) ter apresentado uma acurácia e precisão um pouco maior, o MLP com threshold ajustado para **0,4** obteve maior _Recall_ (0,651 contra 0,559) e maior _F1-score_ (0,619 contra 0,604), mantendo praticamente a mesma _ROC-AUC_ (0,843 contra 0,842). Como o problema é de predição de churn, identificar corretamente clientes propensos ao cancelamento é mais importante do que maximizar a acurácia, tornando o MLP com ajuste de threshold a alternativa mais adequada.

<h3 id="exportacao">Exportação</h3>

#### Modelo

Treinando o modelo com os dados e exportando

In [21]:
mlp_pipeline.fit(X, y)

artefato = {
    "pipeline": mlp_pipeline,
    "threshold": 0.4,
    "model_name": "MLPClassifier"
}

joblib.dump(
    artefato,
    PROJECT_ROOT / "models" / "mlp_classifier_threshold_04.joblib"
)

print("Modelo exportado!")

Modelo exportado!


#### Resultados

Salvando resultados de todos os modelos

In [22]:
resumo.to_csv(PROJECT_ROOT / "reports" / "mlp_resultados.csv", index=False)

print("Resultados salvos!")

Resultados salvos!


Salvando resultados do modelo campeão

In [23]:
metricas_mlp_threshold_04 = resumo[resumo["estrategia"] == "threshold_0.40"][["accuracy", "precision", "recall", "f1", "roc_auc"]]

metricas_mlp_threshold_04.to_json(
    PROJECT_ROOT / "reports" / "metrics" / "metricas_mlp_threshold_04.json",
    orient="records",
    indent=2
)

print("Resultados salvos!")

Resultados salvos!
